In [ ]:
import uproot
import matplotlib.pyplot as plt
import numpy as np
import os

# Define your file paths
file1_path = "~/merged_elecs_test.root"
file2_path = "~/merged_elecs_test_.root"

# Define and create the output directory
output_dir = os.path.expanduser("~/campaign/")
os.makedirs(output_dir, exist_ok=True)

def plot_comparison(h1, h2, name):
    # Extract values and bin edges from the uproot histogram objects
    v1, edges = h1.to_numpy()
    v2, _ = h2.to_numpy()
    if np.sum(v1) > 0:
        v1 = v1 / np.sum(v1)
    if np.sum(v2) > 0:
        v2 = v2 / np.sum(v2)
    # Optional: Skip plotting if both histograms are completely empty
    if np.sum(v1) == 0 and np.sum(v2) == 0:
        return
        
    centers = (edges[:-1] + edges[1:]) / 2

    # Create figure with 2 subplots (main plot and ratio plot)
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(8, 6), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)
    plt.subplots_adjust(hspace=0.05)

    # --- Main Plot ---
    ax1.hist(centers, bins=edges, weights=v1, histtype='step', label='25.12.0', color='blue', linewidth=1.5)
    ax1.hist(centers, bins=edges, weights=v2, histtype='step', label='26.03.1', color='red', linestyle='--', linewidth=1.5)
    ax1.set_ylabel('Events')
    ax1.legend()
    ax1.set_title(name)

    # --- Ratio Plot ---
    # To avoid divide-by-zero warnings, compute the ratio only where v1 is not zero
    ratio = np.full_like(v1, np.nan, dtype=float)
    mask = v1 != 0
    ratio[mask] = v2[mask] / v1[mask]

    ax2.plot(centers, ratio, 'ko', markersize=3)
    ax2.axhline(1.0, color='gray', linestyle='--')
    ax2.set_ylabel('Ratio (F2 / F1)')
    ax2.set_xlabel('Observable Value')

    # Set dynamic y-limits for the ratio plot based on valid data points
    valid_ratio = ratio[~np.isnan(ratio) & ~np.isinf(ratio)]
    if len(valid_ratio) > 0:
        rmin, rmax = np.min(valid_ratio), np.max(valid_ratio)
        if rmax - rmin < 0.1:
            ax2.set_ylim(0.9, 1.1)
        else:
            # Add padding and cap visual outliers to maintain readability
            ax2.set_ylim(max(0.0, rmin - 0.2), min(3.0, rmax + 0.2))

    # --- Save the figure to the specified directory ---
    safe_name = name.replace("/", "_").replace(" ", "_")
    output_path = os.path.join(output_dir, f"{safe_name}.png")
    fig.savefig(output_path, bbox_inches='tight', dpi=150)
    plt.close(fig)

if __name__ == "__main__":
    # Open the ROOT files
    with uproot.open(file1_path) as f1, uproot.open(file2_path) as f2:
        
        # Extract all TH1 keys recursively
        th1_keys = f1.keys(cycle=False, filter_classname=lambda cls: cls.startswith("TH1"), recursive=True)
        
        generated_count = 0
        for key in th1_keys:
            # Ensure the histogram also exists in the second file before trying to plot
            if key in f2:
                plot_comparison(f1[key], f2[key], key)
                generated_count += 1
                
    print(f"Successfully generated and saved {generated_count} comparison plots to {output_dir}")